In [ ]:
#| default_exp prototypical_ex

In [ ]:
#| hide

import plotly.io as pio

In [ ]:
#| hide
pio.renderers.default = "png"

# Prototypical Networks

::: {.callout-note title="Prototypical Networks"}
Prototypical Networks provide a simple way to test whether time‑series classes form meaningful clusters. By learning an embedding space where each class is represented by the mean of its support examples, the method reveals whether samples naturally group around stable prototypes. High classification accuracy and compact prototype geometry indicate that the time series share consistent structure, while diffuse or overlapping prototypes suggest weak or absent clustering.
:::

In [ ]:
#| eval: false
from fhemb.utils.cutils import calc_prototype_accuracy, prepare_data

/Users/radned/.pyenv/versions/p311.fhemb/lib/python3.11/site-packages/threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)
2026-04-19 16:12:44.798747: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
DEBUG:fhemb.config.settings:Loading environment from /Users/radned/.config/fhemb/.env.paths
DEBUG:fhemb.config.s

In [ ]:
#| eval: false
from nbdev_fhemb.wreconstruction_ex import emb_prj_decomp_rec1

 [DEBUG] mixins.wrapper: wrapper of process_subjects called with subjs=[(40, 44), (47, 52), 53, 54, (56, 71), (73, 75)], args=() and kwargs={'title': 'Don Giovanni', 'time_interval': (14200, 15200)}
 [DEBUG] mixins.wrapper: Calling: get_piece_attribute_values with title=Don Giovanni and atributes=['number_of_subjects']
 [DEBUG] mixins.wrapper: to get subjects_int from 'number_of_subjects' record in performance_library
 [DEBUG] mixins.wrapper: Calling: __init__ with subjs=[40, 41, 42, 43, 44, 47, 48, 49, 50, 51, 52, 53, 54, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 73, 74, 75] and args=() and kwargs={'title': 'Don Giovanni', 'time_interval': (14200, 15200)}
 [DEBUG] piece.__init__: Processed subjects: [40, 41, 42, 43, 44, 47, 48, 49, 50, 51, 52, 53, 54, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 73, 74, 75]
 [DEBUG] piece.__init__: Compact numeric subjects: [(40, 44), (47, 54), (56, 71), (73, 75)]
 [INFO] piece._load_from_db: Loading face_t fro

## Define Input Data

In [ ]:
#| eval: false
import numpy as np

#### Stack the time series for respective features along the last axis:

In [ ]:
#| eval: false

X_all=np.stack([ 
    emb_prj_decomp_rec1.get_df_ts_subjs(ftr)[key] for key in emb_prj_decomp_rec1.features.keys() for ftr in emb_prj_decomp_rec1.features[key]
], axis=-1)          

 [DEBUG] embedding.__getattr__: Accessing attribute/method 'features' across embeddings: found in ['percentiles_ts', 'binheights_ts', 'pcomponents_ts', 'features_ts']
 [DEBUG] embedding.__getattr__: Accessing attribute/method 'features' across embeddings: found in ['percentiles_ts', 'binheights_ts', 'pcomponents_ts', 'features_ts']
 [DEBUG] embedding.__getattr__: Accessing attribute/method 'get_df_ts_subjs' across embeddings: found in ['percentiles_ts', 'binheights_ts', 'pcomponents_ts', 'features_ts']
 [DEBUG] embedding.get_df_ts_subjs: Using already calculated values for p20
 [DEBUG] embedding.get_df_ts_subjs: Using already calculated values for p20
 [DEBUG] embedding.get_df_ts_subjs: Using already calculated values for p20
 [DEBUG] embedding.get_df_ts_subjs: Using already calculated values for p20
 [DEBUG] embedding.__getattr__: Accessing attribute/method 'get_df_ts_subjs' across embeddings: found in ['percentiles_ts', 'binheights_ts', 'pcomponents_ts', 'features_ts']
 [DEBUG] embed

In [ ]:
#| eval: false
X_all.shape  # (n_timesteps x n_subjects x n_features)

(1000, 32, 10)

#### Data cleaning, deduplication, NaN removal, and class balancing on two sets of time series data representing different classes

In [ ]:
#| eval: false

X, y = prepare_data(               # prepare the matrix X and label vector y for the two groups
    X_all[:250].transpose(1,2,0),  # first 250 samples for group 0 and transpose to (n_subjects x n_features x n_timesteps)
    X_all[750:].transpose(1,2,0)   # last 250 samples for group 1 and transpose to (n_subjects x n_features x n_timesteps)
)

In [ ]:
#| eval: false
X.shape  # 64 10-dimensional time series 

(64, 10, 250)

## Calculate Classification Accuracy of Prototypical Network 

#### Multidimesional time series

In [ ]:
#| eval: false
calc_prototype_accuracy(   # calculate the accuracy of the prototypical network
    X[y==0],               # samples from group 0
    X[y==1],               # samples from group 1
    num_epochs=100,        # number of training epochs
    lr=0.0005,             # learning rate for the optimizer
    opt_alg='adamw',       # optimization algorithm to use (e.g., 'adam', 'sgd', etc.)
    random_states=10       # number of random train-test-validation splits   
)

 [DEBUG] cutils.train_prototypical_network: Epoch 0, Loss: 0.6918
 [DEBUG] cutils.train_prototypical_network: Epoch 10, Loss: 0.6893
 [DEBUG] cutils.train_prototypical_network: Epoch 20, Loss: 0.6853
 [DEBUG] cutils.train_prototypical_network: Epoch 30, Loss: 0.6794
 [DEBUG] cutils.train_prototypical_network: Epoch 40, Loss: 0.6738
 [DEBUG] cutils.train_prototypical_network: Epoch 50, Loss: 0.6732
 [DEBUG] cutils.train_prototypical_network: Epoch 60, Loss: 0.6727
 [DEBUG] cutils.train_prototypical_network: Epoch 70, Loss: 0.6722
 [DEBUG] cutils.train_prototypical_network: Epoch 80, Loss: 0.6695
 [DEBUG] cutils.train_prototypical_network: Epoch 90, Loss: 0.6501
 [INFO] cutils.train_prototypical_network: Epoch 99, Loss: 0.5974
 [INFO] cutils.train_prototypical_network: Accuracy: 0.8000
 [INFO] cutils.evaluate_prototypical_network: Test Accuracy: 66.67%
 [DEBUG] cutils.train_prototypical_network: Epoch 0, Loss: 0.6903
 [DEBUG] cutils.train_prototypical_network: Epoch 10, Loss: 0.6850
 [DE

[0.6666666666666666,
 0.3333333333333333,
 0.6666666666666666,
 0.6666666666666666,
 1.0,
 0.3333333333333333,
 0.6666666666666666,
 0.6666666666666666,
 0.6666666666666666,
 0.0]

#### 1-dimensiona time series

In [ ]:
#| eval: false

features_all =[ftr for key in emb_prj_decomp_rec1.features.keys() for ftr in emb_prj_decomp_rec1.features[key]]
features_all[2]

 [DEBUG] embedding.__getattr__: Accessing attribute/method 'features' across embeddings: found in ['percentiles_ts', 'binheights_ts', 'pcomponents_ts', 'features_ts']
 [DEBUG] embedding.__getattr__: Accessing attribute/method 'features' across embeddings: found in ['percentiles_ts', 'binheights_ts', 'pcomponents_ts', 'features_ts']
 [DEBUG] embedding.__getattr__: Accessing attribute/method 'features' across embeddings: found in ['percentiles_ts', 'binheights_ts', 'pcomponents_ts', 'features_ts']
 [DEBUG] embedding.__getattr__: Accessing attribute/method 'features' across embeddings: found in ['percentiles_ts', 'binheights_ts', 'pcomponents_ts', 'features_ts']
 [DEBUG] embedding.__getattr__: Accessing attribute/method 'features' across embeddings: found in ['percentiles_ts', 'binheights_ts', 'pcomponents_ts', 'features_ts']


'bh1'

In [ ]:
#| eval: false
calc_prototype_accuracy(           # calculate the accuracy of the prototypical network
    X[:,2,:][y==0],                # samples from group 0 
    X[:,2,:][y==1],                # samples from group 1
    num_epochs=500,                # number of training epochs
    lr=0.0005,                     # learning rate for the optimizer
    opt_alg='adamw',               # optimization algorithm to use (e.g., 'adam', 'sgd', etc.)
    random_states=10               # number of random train-test-validation splits
)

 [DEBUG] cutils.train_prototypical_network: Epoch 0, Loss: 0.6932
 [DEBUG] cutils.train_prototypical_network: Epoch 10, Loss: 0.6931
 [DEBUG] cutils.train_prototypical_network: Epoch 20, Loss: 0.6931
 [DEBUG] cutils.train_prototypical_network: Epoch 30, Loss: 0.6931
 [DEBUG] cutils.train_prototypical_network: Epoch 40, Loss: 0.6931
 [DEBUG] cutils.train_prototypical_network: Epoch 50, Loss: 0.6931
 [DEBUG] cutils.train_prototypical_network: Epoch 60, Loss: 0.6931
 [DEBUG] cutils.train_prototypical_network: Epoch 70, Loss: 0.6931
 [DEBUG] cutils.train_prototypical_network: Epoch 80, Loss: 0.6931
 [DEBUG] cutils.train_prototypical_network: Epoch 90, Loss: 0.6931
 [DEBUG] cutils.train_prototypical_network: Epoch 100, Loss: 0.6931
 [DEBUG] cutils.train_prototypical_network: Epoch 110, Loss: 0.6931
 [DEBUG] cutils.train_prototypical_network: Epoch 120, Loss: 0.6931
 [DEBUG] cutils.train_prototypical_network: Epoch 130, Loss: 0.6931
 [DEBUG] cutils.train_prototypical_network: Epoch 140, Loss

[0.6666666666666666,
 0.3333333333333333,
 0.3333333333333333,
 0.0,
 0.6666666666666666,
 0.0,
 0.6666666666666666,
 0.6666666666666666,
 0.6666666666666666,
 0.6666666666666666]